# Observability-Guided Head Pruning — Kaggle GPU Sweep

This notebook runs the full pruning sweep (methods: **observability** + **hybrid**) on a Kaggle GPU.

**Before run, in the Kaggle UI (right-hand panel):**
1. **Accelerator → GPU** (T4 x2 or P100)
2. **Internet → On** (needs a phone-verified account)
3. Upload the project as a **Dataset**: zip the project and add it via *Add Input → Datasets → Upload*.

To run unattended for up to 12 h (survives closing the tab), use **Save Version → Save & Run All (Commit)**. The sweep is resumable, so a re-run skips finished cells.

## Step 1 — Locate and copy the project
`/kaggle/input` is read-only, so we copy the project into the writable `/kaggle/working`. This auto-detects the folder containing `scripts/run_pruning.py`. It also checks what accelerator we are using

In [ ]:
import glob, os, shutil, sys, torch

cands = glob.glob("/kaggle/input/**/scripts/run_pruning.py", recursive=True)
assert cands, "Could not find scripts/run_pruning.py under /kaggle/input — check your Dataset upload."
SRC = os.path.dirname(os.path.dirname(cands[0]))
ROOT = "/kaggle/working/project"
shutil.copytree(SRC, ROOT, dirs_exist_ok=True)
sys.path.insert(0, ROOT)

print("Found project at:", SRC)
print("Copied to       :", ROOT)
print("CUDA available  :", torch.cuda.is_available(), "->",
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

## Step 2 — Install dependencies
We install only `transformers` and `datasets` at *your* pinned versions (from `requirements.txt`) so the dataset-loading behaviour matches your local env. We deliberately **do not** touch `torch` — Kaggle's torch is the CUDA build.

In [ ]:
import re, subprocess, sys
reqs = open(f"{ROOT}/requirements.txt").read().splitlines()
want = [r.strip() for r in reqs if re.match(r"^\s*(transformers|datasets)\b", r)]
print("Installing:", want or "(none found — set manually if needed)")
if want:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *want], check=True)

## Step 3 — Check for errors
Verifies there are no errors

In [ ]:
issues = []

try:
    import importlib
    importlib.import_module("src.methods.observability")
except Exception as e:
    issues.append(f"importing src.methods.observability failed: {e}")

print("PREFLIGHT:", "OK" if not issues else "PROBLEMS FOUND")
for i in issues:
    print("  -", i)
if issues:
    print("\nFix these in your local folder, re-zip, replace the Kaggle Dataset, and re-run.")

## Step 4 — Sanity run
Confirm a single run completes end-to-end (loads the checkpoint, prunes, fine-tunes, evaluates) before launching the whole sweep.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "scripts/run_pruning.py",
                "--dataset", "imdb", "--method", "observability", "--sparsity", "0.42"],
               cwd=ROOT)

## Step 5 — Full sweep
Two observability methods x 3 datasets x 4 sparsities = 24 runs.

In [ ]:
import subprocess, itertools, os, sys
os.makedirs(f"{ROOT}/results/sweep_logs", exist_ok=True)
METHODS  = ["observability", "hybrid"]
DATASETS = ["imdb", "ag_news", "banking77"]
SPARS    = [0.20, 0.30, 0.42, 0.50]

for method, dataset, sp in itertools.product(METHODS, DATASETS, SPARS):
    log = f"{ROOT}/results/sweep_logs/{method}_{dataset}_{sp}.log"
    if os.path.exists(log) and open(log).read().count("Acc:") >= 2:
        print("skip", method, dataset, sp); continue
    print(f"=== RUN {method} {dataset} {sp} ===", flush=True)
    with open(log, "w") as f:
        p = subprocess.Popen([sys.executable, "scripts/run_pruning.py",
                              "--dataset", dataset, "--method", method, "--sparsity", str(sp)],
                             cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                             text=True, bufsize=1)
        for line in p.stdout:
            print(line, end=""); f.write(line)
        p.wait()
print("\nSweep complete.")

## Step 6 — Build the results table
Parses the logs into a table and writes `results/sweep_results.csv`

In [ ]:
import re, glob, os, pandas as pd
METRIC = re.compile(r"Acc:\s*([\d.]+)%.*?F1:\s*([\d.]+)%.*?Compression:\s*([\d.]+)%")
DATASETS = ["imdb", "ag_news", "banking77"]
rows = []
for log in glob.glob(f"{ROOT}/results/sweep_logs/*.log"):
    name = os.path.basename(log)[:-4]
    m = re.search(r"_([01]\.\d+)$", name); sp = float(m.group(1)); rest = name[:m.start()]
    ds = next(d for d in DATASETS if rest.endswith(d)); method = rest[:-(len(ds)+1)]
    lines = [(ln, METRIC.search(ln)) for ln in open(log).read().splitlines() if METRIC.search(ln)]
    base   = next((float(mm.group(1)) for ln, mm in lines if "baseline" in ln.lower()), None)
    pruned = [(ln, mm) for ln, mm in lines if "baseline" not in ln.lower()]
    if pruned:
        mm = pruned[-1][1]
        rows.append(dict(method=method, dataset=ds, sparsity=sp, baseline=base,
                         acc=float(mm.group(1)), f1=float(mm.group(2)),
                         param_drop_pct=float(mm.group(3)),
                         acc_drop=round(base - float(mm.group(1)), 2) if base else None))
    else:
        rows.append(dict(method=method, dataset=ds, sparsity=sp, acc=None, note="MISSING — rerun"))

df = pd.DataFrame(rows).sort_values(["dataset", "method", "sparsity"]).reset_index(drop=True)
df.to_csv(f"{ROOT}/results/sweep_results.csv", index=False)
print("Wrote", f"{ROOT}/results/sweep_results.csv")
df

**Download the results:** the CSV and logs are under `/kaggle/working/project/results/`. After a committed run they appear in the notebook version's *Output* tab. These are the numbers into Table 3 of the paper.